<a href="https://colab.research.google.com/github/goswami-p/vallabhavani-reader/blob/main/notebooks/fantasy-vox-train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fantasy Storyteller voice clone + Piper fine-tune
1. XTTS-v2 zero-shot clone from 14.5s reference
2. Synthetic narration corpus (~1000-1200 sentences)
3. Fine-tune Piper medium (en_GB base) with piper1-gpl
4. Export ONNX for in-browser use

XTTS is CPML-licensed (non-commercial). This is a personal reading site.


In [1]:
# Colab + Kaggle both: create the work dir the notebook expects (Colab runs as root)
import os
os.makedirs('/kaggle/working', exist_ok=True)
print('on colab:', bool(os.environ.get('COLAB_RELEASE_TAG')))
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO-GPU')
assert torch.cuda.is_available(), 'No GPU — on Colab: Runtime > Change runtime type > T4 GPU'


on colab: True
cuda: True Tesla T4


In [2]:
# fail-fast: GPU + internet must be ON before anything long runs
import sys, torch, urllib.request
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO-GPU')
assert torch.cuda.is_available(), 'GPU is OFF — fix kernel metadata (enable_gpu)'
urllib.request.urlopen('https://github.com', timeout=15)
print('net OK')


python 3.13.15
torch 2.11.0+cu130 cuda True Tesla T4
net OK


In [3]:
!apt-get -qq update >/dev/null 2>&1
!apt-get -qq install -y espeak-ng libsndfile1 build-essential cmake ninja-build git ffmpeg >/dev/null 2>&1
!espeak-ng --version
!git --version


eSpeak NG text-to-speech: 1.51  Data at: /usr/lib/x86_64-linux-gnu/espeak-ng-data
git version 2.43.0


In [4]:
# Piper training stack (piper1-gpl, Apache-2.0) — no MFA needed (espeak phonemize)
!git clone -q --depth 1 https://github.com/OHF-Voice/piper1-gpl.git /kaggle/working/piper1-gpl
%cd /kaggle/working/piper1-gpl
!pip install -q -e '.[train]' 2>&1 | tail -5
!./build_monotonic_align.sh 2>&1 | tail -3
!python3 setup.py build_ext --inplace 2>&1 | tail -3
!pip install -q coqui-tts 2>&1 | tail -5
import torch, TTS
print('OK torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'TTS', TTS.__version__ if hasattr(TTS, '__version__') else 'ok')
assert torch.cuda.is_available(), 'GPU lost after installs — do not continue'


/kaggle/working/piper1-gpl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.9/803.9 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 60.7 MB/s eta 0:00:00
	2. Use an 'int' return type on 'maximum_path_each' to allow an error code to be returned.
Compiling /kaggle/working/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx because it changed.
[1/1] Cythonizing /kaggle/working/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx
  File "/kaggle/working/piper1-gpl/setup.py", line 6, in <module>
    from skbuild import setup
ModuleNotFoundError: No module named 'skbuild'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 62.7 MB/s 

In [5]:
# reference voice (embedded)
import base64, pathlib
p = pathlib.Path('/kaggle/working/reference.mp3')
p.write_bytes(base64.b64decode('...'))
print('reference:', p.stat().st_size, 'bytes')


reference: 586350 bytes


In [ ]:
# XTTS-v2 (~2GB download on first run)
from huggingface_hub import snapshot_download
xtts_dir = snapshot_download('coqui/XTTS-v2', local_dir='/kaggle/working/xtts')
print('xtts at', xtts_dir)
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
import torch
cfg = XttsConfig(); cfg.load_json('/kaggle/working/xtts/config.json')
model = Xtts.from_config(cfg)
model.load_checkpoint(cfg, checkpoint_dir='/kaggle/working/xtts')
model = model.cuda().eval()
gpt_lat, spk_lat = model.get_conditioning_latents(audio_path=['/kaggle/working/reference.mp3'], max_ref_length=30, sound_norm_refs=True)
print('latents ok', gpt_lat.shape, spk_lat.shape)


In [ ]:
# narration corpus: public-domain story text (Gutenberg) + hand-written fantasy lines
import re, random
random.seed(7)
fantasy_lines = [
    'The old lighthouse keeper climbed the spiral stairs one final time, her lamp cutting a pale seam through the fog.',
    'Beyond the iron gates, the forest held its breath, and the moon laid a silver road between the trees.',
    'He spoke the words slowly, as every keeper had for a hundred years, and the deep answered with a low and ancient hum.',
    'Maps of forgotten kingdoms lay spread across the table, their edges brown with age and adventure.',
    'She had heard the song before, in dreams, rising from somewhere beneath the cold black water of the lake.',
    'The dragon did not roar. It simply opened one amber eye, and the village knew its bargain had been remembered.',
    'Snow fell on the mountain passes, soft and endless, burying the road that led to the last free city.',
    'In the library of the dead king, every book was chained to its shelf except one.',
    'The sword was not magical. The arm that carried it had simply refused, for thirty years, to forget its oath.',
    'Wind moved through the ruined chapel, turning the pages of a hymnal no one had sung in living memory.',
    'They set out at dawn with little food and less hope, and the road ahead was longer than any of them had dreamed.',
    'The harbour town woke to bells, to the smell of tar and bread, and to the news that the ships were coming home.',
    'What the old woman told him that night by the fire, he repeated to no one, and carried like a stone ever after.',
    'The forest path split in two, and neither choice looked kind, so he sat down beneath the oak and waited for the stars.',
    'Legends say the tower was built by giants. The truth was quieter, and better, and no one remembered it.',
    'She wrote the letter three times and sent it once, and the words she finally chose were fewer than the ones she burned.',
    'The river carried their small boat under a bridge of grey stone, and onward toward a country none of them had seen.',
    'Every winter the village left a bowl of milk on the step, and every spring the wolves stayed west of the fields.',
    'He was not a brave man, but the child was behind him, so he stood his ground and raised the old lantern high.',
    'The stars over the desert were so thick and bright that travellers navigated by them, as sailors do by the sea.',
    'A single candle burned in the window of the hermitage, and those who saw it from the road knew someone was still praying.',
    'The book of hours was written in three hands: the scribe, the illuminator, and, centuries later, a small careful child.',
    'When the bells rang at midnight, the baker crossed himself, the guard reached for his spear, and the cat slept on.',
    'The meadow lay full of hares and silence, and the two travellers walked through it speaking little, thinking much.',
    'It was the kind of rain that made the whole world small: the road, the horse, the roof of the wayside inn.',
    'The king asked for truth, and was given it, and never asked that particular question of that particular advisor again.',
    'On clear nights, if you climbed the eastern hill, you could see the lights of the fishing fleet winking far out on the water.',
    'The monk copied the old text by candlelight, letter by letter, not knowing he was saving half a language.',
    'They found the gate ajar, the garden wild within it, and the sundial still pointing faithfully at an hour no one now uses.',
    'The shepherd counted his flock twice, as his father had taught him, and found one sheep missing and one mystery more.',
]
urls = [
    'https://www.gutenberg.org/cache/epub/1727/pg1727.txt',      # Odyssey, book 1 excerpt vibes
    'https://www.gutenberg.org/cache/epub/16317/pg16317.txt',    # Bulfinch's mythology
    'https://www.gutenberg.org/cache/epub/11/pg11.txt',         # Alice
    'https://www.gutenberg.org/cache/epub/51898/pg51898.txt',   # Norse stories
]
texts = []
for u in urls:
    try:
        import urllib.request
        raw = urllib.request.urlopen(u, timeout=60).read().decode('utf-8', 'ignore')
        raw = re.sub(r'\r', '', raw)
        raw = re.sub(r'\[?(Project Gutenberg[^\]]*|START OF (THE|THIS) PROJECT GUTENBERG[^\n]*|END OF (THE|THIS) PROJECT GUTENBERG[^\n]*)\]?', ' ', raw, flags=re.I)
        texts.append(raw)
        print('fetched', u, len(raw))
    except Exception as e:
        print('skip', u, e)
blob = '\n'.join(texts)
sents = []
for m in re.finditer(r'([A-Z][^.!?।\n]{25,140}[.!?])', blob):
    s = re.sub(r'\s+', ' ', m.group(1)).strip()
    sents.append(s)
sents = [s for s in sents if not s.startswith(('http', 'www', 'Gutenberg', 'Chapter'))]
random.shuffle(sents)
seen = set(); final = []
for s in fantasy_lines + sents:
    k = s.lower()
    if k in seen or len(s) < 25 or len(s) > 150: continue
    seen.add(k); final.append(s)
    if len(final) >= 1200: break
print('corpus sentences:', len(final))
pathlib.Path('/kaggle/working/sentences.txt').write_text('\n'.join(final), encoding='utf-8')


In [ ]:
# synth corpus with cloned voice (resampled to 22050 Hz mono, Piper medium format)
import pathlib, torch, torchaudio, csv, time
out = pathlib.Path('/kaggle/working/corpus'); (out / 'wavs').mkdir(parents=True, exist_ok=True)
sents = pathlib.Path('/kaggle/working/sentences.txt').read_text(encoding='utf-8').splitlines()
done = set(int(p.stem[3:]) for p in (out / 'wavs').glob('utt*.wav'))
t0 = time.time()
for i, txt in enumerate(sents, 1):
    if i in done: continue
    try:
        wav = model.inference(txt, 'en', gpt_lat, spk_lat, temperature=0.65, top_k=50, top_p=0.85, enable_text_splitting=True)['wav']
        w = torch.tensor(wav).unsqueeze(0)
        w = torchaudio.functional.resample(w, 24000, 22050)
        torchaudio.save(str(out / 'wavs' / f'utt{i:04d}.wav'), w, 22050)
    except Exception as e:
        print('fail', i, e); continue
    if i % 50 == 0:
        el = time.time() - t0
        print(f'{i}/{len(sents)}  {el/60:.1f} min', flush=True)
rows = []
for i, txt in enumerate(sents, 1):
    fp = out / 'wavs' / f'utt{i:04d}.wav'
    if fp.exists(): rows.append(f'wavs/utt{i:04d}.wav|{txt}')
(out / 'metadata.csv').write_text('\n'.join(rows), encoding='utf-8')
print('corpus ready:', len(rows), 'utterances')


In [ ]:
# base Piper checkpoint (fine-tune from it — much faster + stabler)
from huggingface_hub import list_repo_files, hf_hub_download
files = list_repo_files('rhasspy/piper-checkpoints', repo_type='dataset')
cands = [f for f in files if f.startswith('en/en_GB/northern_english_male/medium/') and f.endswith(('.ckpt', '.pt'))]
cands.sort()
print('candidates tail:', cands[-5:])
best = cands[-1]
print('using', best)
ckpt = hf_hub_download('rhasspy/piper-checkpoints', best, repo_type='dataset', local_dir='/kaggle/working/ckpts')
print('base ckpt at', ckpt)
pathlib.Path('/kaggle/working/base_ckpt.txt').write_text(ckpt)


In [ ]:
# fine-tune Piper (medium, espeak en-gb-x-rp). ~60-90 min for 150 epochs on T4.
import subprocess, pathlib, sys
ckpt = pathlib.Path('/kaggle/working/base_ckpt.txt').read_text().strip()
log = open('/kaggle/working/train.log', 'w')
cmd = [sys.executable, '-m', 'piper.train', 'fit',
       '--data.voice_name', 'en_GB-fantasy-medium',
       '--data.csv_path', '/kaggle/working/corpus/metadata.csv',
       '--data.audio_dir', '/kaggle/working/corpus',
       '--model.sample_rate', '22050',
       '--data.espeak_voice', 'en-gb-x-rp',
       '--data.cache_dir', '/kaggle/working/train_cache',
       '--data.config_path', '/kaggle/working/en_GB-fantasy-medium.onnx.json',
       '--data.batch_size', '16',
       '--data.num_workers', '2',
       '--trainer.max_epochs', '150',
       '--trainer.log_every_n_steps', '10',
       '--trainer.enable_checkpointing', 'True',
       '--ckpt_path', ckpt]
print(' '.join(cmd))
proc = subprocess.Popen(cmd, cwd='/kaggle/working/piper1-gpl', stdout=log, stderr=subprocess.STDOUT)
rc = proc.wait()
print('train exit', rc)
!tail -20 /kaggle/working/train.log
assert rc == 0, 'training failed — see /kaggle/working/train.log'


In [ ]:
# export ONNX + package
import glob, pathlib, shutil, subprocess, sys, zipfile
ckpts = glob.glob('/kaggle/working/piper1-gpl/lightning_logs/**/checkpoints/*.ckpt', recursive=True)
ckpts.sort(key=lambda p: pathlib.Path(p).stat().st_mtime)
assert ckpts, 'no lightning checkpoint found'
newest = ckpts[-1]
print('exporting', newest)
rc = subprocess.call([sys.executable, '-m', 'piper.train.export_onnx',
                      '--checkpoint', newest,
                      '--output-file', '/kaggle/working/en_GB-fantasy-medium.onnx'],
                     cwd='/kaggle/working/piper1-gpl')
assert rc == 0, 'onnx export failed'
outdir = pathlib.Path('/kaggle/working/output'); outdir.mkdir(exist_ok=True)
shutil.copy('/kaggle/working/en_GB-fantasy-medium.onnx', outdir / 'en_GB-fantasy-medium.onnx')
cfgp = pathlib.Path('/kaggle/working/en_GB-fantasy-medium.onnx.json')
if cfgp.exists(): shutil.copy(cfgp, outdir / 'en_GB-fantasy-medium.onnx.json')
else:
    cands = glob.glob('/kaggle/working/**/en_GB-fantasy-medium*.json', recursive=True)
    print('config candidates', cands)
    if cands: shutil.copy(cands[0], outdir / 'en_GB-fantasy-medium.onnx.json')
with zipfile.ZipFile(outdir / 'fantasy-voice.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in outdir.iterdir():
        if f.suffix != '.zip': z.write(f, f.name)
print('packaged:', list(outdir.iterdir()))
!ls -lh /kaggle/working/output
